In [1]:
# Imports

import os
import rdkit
import logging
import warnings
import subprocess
import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm
from __future__ import division
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display
from pandarallel import pandarallel
from __future__ import absolute_import
from __future__ import unicode_literals
from rdkit.Chem.Draw import IPythonConsole
from dimorphite_dl import protonate_smiles
from rdkit.Chem.EState import Fingerprinter
from mordred import Calculator, descriptors
from rdkit.DataStructs import BitVectToText
from rdkit.Chem.MACCSkeys import GenMACCSKeys
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from rdkit.ML.Descriptors import MoleculeDescriptors 
from sklearn.model_selection import train_test_split
from rdkit import Chem, DataStructs, rdBase, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import AllChem, Draw, Descriptors, PandasTools, AddHs, inchi, MolStandardize


# Suppress warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

In [2]:
# Define directories and data

root_dir = '/Users/HQ/Repositories/DILI/'
data_dir = os.path.join(root_dir, 'Data')

model_tox_data = os.path.join(data_dir, 'Alloriginaldata_19911.csv')
model_tox_data = pd.read_csv(model_tox_data)

In [3]:
# Check for invalid entries

# Initialize counters for each criterion
counters = {
    'invalid_smiles': 0,
    'no_carbon': 0,
    'is_mixture': 0,
    'contains_metals': 0,
    'above_mw_threshold': 0
}

# Function to check exclusion criteria and update counters
def check_criteria(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles) 
        if mol is None:
            counters['invalid_smiles'] += 1
            return False
        if not mol.HasSubstructMatch(Chem.MolFromSmarts('C')):
            counters['no_carbon'] += 1
            return False
        if mol.GetNumAtoms() != Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)[0].GetNumAtoms():
            counters['is_mixture'] += 1
            return False
        if any(atom.GetSymbol().isupper() and atom.GetSymbol() not in ['C', 'H', 'N', 'O', 'P', 'S', 'F', 'Cl', 'Br', 'I'] for atom in mol.GetAtoms()):
            counters['contains_metals'] += 1
            return False
        if Descriptors.MolWt(mol) > 1500:
            counters['above_mw_threshold'] += 1
            return False
        return True
    except:
        counters['invalid_smiles'] += 1
        return False

# Apply the criteria checking function to each SMILES string
model_tox_data['include'] = model_tox_data['smiles_r'].apply(check_criteria)

# Filter the DataFrame to include only the molecules that passed all criteria
valid_tox_data = model_tox_data[model_tox_data['include']]

# Now, print the counters to see how many compounds were excluded for each reason
for criterion, count in counters.items():
    print(f"{criterion}: {count}")

# The result is 'valid_tox_data' which contains only the molecules meeting the criteria
# You can now drop the 'include' column if you want
valid_tox_data = valid_tox_data.drop(columns=['include'])

invalid_smiles: 0
no_carbon: 1153
is_mixture: 1072
contains_metals: 14
above_mw_threshold: 43


In [4]:
# Standardize Entries with RDKit

def standardize_jumpcp(smiles):
    
    # Read SMILES and convert it to RDKit mol object
    mol = Chem.MolFromSmiles(smiles)
 
    smiles_clean_counter = Counter()
    mol_dict = {}
    is_finalize = False

    try:
        for _ in range(5):

            mol = Chem.MolFromSmiles(smiles)
            inchi_standardised = Chem.MolToInchi(mol)
            inchi_standardised
            mol = Chem.MolFromInchi(inchi_standardised)
            mol = rdMolStandardize.Cleanup(mol) 
            mol = rdMolStandardize.FragmentParent(mol) # if many fragments, get the "parent" (the actual mol we are interested in) 
            uncharger = rdMolStandardize.Uncharger()

            mol = uncharger.uncharge(mol) # standardize molecules using MolVS and RDKit
            mol = rdMolStandardize.ChargeParent(mol)
            mol = rdMolStandardize.IsotopeParent(mol)
            mol = rdMolStandardize.StereoParent(mol)

            # Normalize tautomers 
            # Method 1
            normalizer = rdMolStandardize.TautomerEnumerator()
            mol = normalizer.Canonicalize(mol)

            # Method 2
            te = rdMolStandardize.TautomerEnumerator() # idem
            mol = te.Canonicalize(mol)

            # Method 3
            mol = rdMolStandardize.TautomerParent(mol)

            # Final Rules
            mol = rdMolStandardize.Normalize(mol)
            mol_standardized = mol

            # Convert mol object back to SMILES
            smiles_standardized = Chem.MolToSmiles(mol_standardized)

            if smiles == smiles_standardized:
                is_finalize = True
                break

            smiles_clean_counter[smiles_standardized] += 1
            if smiles_standardized not in mol_dict:
                mol_dict[smiles_standardized] = mol_standardized

            smiles = smiles_standardized
            mol = Chem.MolFromSmiles(smiles)

        if not is_finalize:
            # If the standardization process is not finalized, we choose the most common SMILES from the counter and the corresponding mol object
            smiles_standardized = smiles_clean_counter.most_common()[0][0]
            mol_standardized = mol_dict[smiles_standardized]
            
        return smiles_standardized
    except Exception as e:
        print(f'{smiles}: {e}')
        return 'INVALID'

tqdm.pandas()
pandarallel.initialize(progress_bar=True)
valid_tox_data["smiles_r"] = valid_tox_data["smiles_r"].parallel_apply(standardize_jumpcp)
valid_tox_data_file = os.path.join(data_dir, 'valid_tox_data.csv')
valid_tox_data.to_csv(valid_tox_data_file)

INFO: Pandarallel will run on 10 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


*OC1C(O)C(CO)OC(OCC2OC(OC3C(O)C(CO)OC(OC4C(O)C(COC5OC(CO)C(O)C(O)C5O)OC(OC5C(O)C(*)OC(CO)C5O)C4O)C3O)C(O)C(OC3OC(CO)C(O)C(O)C3O)C2O)C1O: Molecule is None
*OC1OC(C(=O)O)C(OC2OC(CO)C(O)C(O*)C2NC(C)=O)C(O)C1O: Molecule is None


In [5]:
# Protonate entries with DimorphiteDL

def protonate(smiles):

    protonated_smiles = protonate_smiles(
    smiles,
    ph_min=7.0,
    ph_max=7.0,
    precision=0,
    max_variants=1
)
    if len(protonated_smiles) > 0:
                protonated_smiles = protonated_smiles[0]
    return protonated_smiles
    
def smiles_to_inchikey(smiles):

    try:
        mol = Chem.MolFromSmiles(smiles)
        inchi_string = Chem.MolToInchi(mol)
        inchi_key = Chem.inchi.InchiToInchiKey(inchi_string)
        return inchi_key
    
    except Exception as e:
        print(f'{smiles}: {e}')
        return "INVALID"

valid_tox_data['InChIKey'] = valid_tox_data["smiles_r"].parallel_apply(smiles_to_inchikey)
valid_tox_data['InChIKey14'] = valid_tox_data["InChIKey"].str[:14]
valid_tox_data['protonated_smiles_r'] = valid_tox_data["smiles_r"].parallel_apply(protonate)

INVALID: Python argument types in
    rdkit.Chem.rdinchi.MolToInchi(NoneType, str)
did not match C++ signature:
    MolToInchi(RDKit::ROMol mol, std::__1::basic_string<char, std::__1::char_traits<char>, std::__1::allocator<char>> options='')
INVALID: Python argument types in
    rdkit.Chem.rdinchi.MolToInchi(NoneType, str)
did not match C++ signature:
    MolToInchi(RDKit::ROMol mol, std::__1::basic_string<char, std::__1::char_traits<char>, std::__1::allocator<char>> options='')


In [6]:
# Group entries by source and deduplicate

grouped_tox_data = valid_tox_data.groupby('Source_rank')

# Placeholder for processed groups
processed_groups = []

for _, group in grouped_tox_data:
    # Step 2: Sort within each group by 'TOXICITY' in descending order (1s before 0s)
    # and by 'InChIKey14' to ensure consistency in duplicate removal
    sorted_group = group.sort_values(by=['TOXICITY', 'InChIKey14'], ascending=[False, True])
    
    # Step 3: Drop duplicates based on 'InChIKey14', keeping the first (the toxic one if it exists)
    deduped_group = sorted_group.drop_duplicates(subset='InChIKey14', keep='first')
    
    # Collecting processed groups
    processed_groups.append(deduped_group)

# Step 4: Combine the processed groups back into a single DataFrame
grouped_tox_data = pd.concat(processed_groups, ignore_index=True)

# Display or return the processed DataFrame
grouped_tox_data = grouped_tox_data[grouped_tox_data["smiles_r"]!="INVALID"]
grouped_tox_data = grouped_tox_data[grouped_tox_data["InChIKey"]!="INVALID"]
grouped_tox_data = grouped_tox_data[grouped_tox_data["protonated_smiles_r"]!="INVALID"]


In [7]:
# Define train and test data

train_data_tox = grouped_tox_data[grouped_tox_data["Data"]!="DILI"]
train_data_tox["Data"] = "LivTox"

outer_test_data_tox = grouped_tox_data[grouped_tox_data["Data"]=="DILI"]
outer_test_data_tox = outer_test_data_tox.sort_values(["TOXICITY"], ascending=False)
outer_test_data_tox = outer_test_data_tox.drop_duplicates(subset=["InChIKey14"], keep="first").reset_index(drop=True)
outer_test_data_tox = outer_test_data_tox[outer_test_data_tox["smiles_r"]!="INVALID"]
outer_test_data_tox["Data"] = "DILI"

outer_test_data_tox_file = os.path.join(data_dir, 'DILI_Goldstandard.csv')
outer_test_data_tox.to_csv(outer_test_data_tox_file, index=False)

final_tox_data = pd.concat([train_data_tox, outer_test_data_tox]).reset_index(drop=True)

In [ ]:
# Generate Morgan Fingerprints

def MorganFingerprint(s):
    x = Chem.MolFromSmiles(s)
    return (AllChem.GetMorganFingerprintAsBitVect(x,2,2048))

MorganFingerprint_array = np.stack(final_tox_data['protonated_smiles_r'].apply(MorganFingerprint))

Morgan_fingerprint_collection = []
for x in np.arange(MorganFingerprint_array.shape[1]):
    x = "Mfp"+str(x)
    Morgan_fingerprint_collection.append(x)

Morgan_fingerprint_table = pd.DataFrame(MorganFingerprint_array, columns=Morgan_fingerprint_collection)

In [ ]:
# Generate MACCS Keys

def MACCSKeysFingerprint(s):
    x = Chem.MolFromSmiles(s)
    return (AllChem.GetMACCSKeysFingerprint(x))

MACCSfingerprint_array = np.stack(final_tox_data['protonated_smiles_r'].apply(MACCSKeysFingerprint))

MACCS_collection = []
for x in np.arange(MACCSfingerprint_array.shape[1]):
    x = "MACCS"+str(x)
    MACCS_collection.append(x)

MACCSfingerprint_table = pd.DataFrame(MACCSfingerprint_array, columns=MACCS_collection)

In [ ]:
# Generate Mordred Table

calc = Calculator(descriptors, ignore_3D=True)

print(len(calc.descriptors))

Ser_Mol = final_tox_data['protonated_smiles_r'].apply(Chem.MolFromSmiles)
Mordred_table = calc.pandas(Ser_Mol)
Mordred_table = Mordred_table.astype('float')

Mordred_table_file = os.path.join(data_dir, 'data/Mordred_table.csv')
Mordred_table.to_csv(Mordred_table)

Mordred_table = Mordred_table.dropna(axis='columns')

In [ ]:
# Generate 2D descriptors

def get_num_charged_atoms_neg(mol):
    mol_h = Chem.AddHs(mol)
    Chem.rdPartialCharges.ComputeGasteigerCharges(mol_h)
     
    positive = 0
    negative = 0
     
    for atom in mol_h.GetAtoms():
        if float(atom.GetProp('_GasteigerCharge')) <= 0:
            negative += 1
     
    return negative

def get_num_charged_atoms_pos(mol):
    mol_h = Chem.AddHs(mol)
    Chem.rdPartialCharges.ComputeGasteigerCharges(mol_h)
     
    positive = 0
    negative = 0
     
    for atom in mol_h.GetAtoms():
        if float(atom.GetProp('_GasteigerCharge')) >= 0:
            positive += 1
    return positive

def get_assembled_ring(mol):
    ring_info = mol.GetRingInfo()
    num_ring = ring_info.NumRings()
    ring_atoms = ring_info.AtomRings()
    num_assembled = 0
     
    for i in range(num_ring):
        for j in range(i+1, num_ring):
            x = set(ring_atoms[i])
            y = set(ring_atoms[j])
            if not x.intersection(y): # 2つの環が縮環でない場合に
                for x_id in x:
                    x_atom = mol.GetAtomWithIdx(x_id)
                    neighbors = [k.GetIdx() for k in x_atom.GetNeighbors()]
                    for x_n in neighbors:
                        if x_n in y: # 環同士を繋ぐ結合があるか否か
                            num_assembled += 1
     
    return num_assembled

def get_num_stereocenters(mol):
    return AllChem.CalcNumAtomStereoCenters(mol) + AllChem.CalcNumUnspecifiedAtomStereoCenters(mol)

def calc_descriptors(dataframe):
    mols = dataframe.protonated_smiles_r.apply(Chem.MolFromSmiles)
    descr = []
    for m in mols:
        descr.append([ Descriptors.TPSA(m),
               Descriptors.NumRotatableBonds(m),
               AllChem.CalcNumRings(m),
               Descriptors.NumAromaticRings(m),
               Descriptors.NumHAcceptors(m),
               Descriptors.NumHDonors(m),
               Descriptors.FractionCSP3(m),
               Descriptors.MolLogP(m) ,
               Descriptors.NHOHCount(m),
               Descriptors.NOCount(m),
               Descriptors.NumHeteroatoms(m),
               get_num_charged_atoms_pos(m),
               get_num_charged_atoms_neg(m),
               get_assembled_ring(m),
               get_num_stereocenters(m)])
    descr = np.asarray(descr)
    return(descr)

descs = [ 'PSA', 'n_rot_bonds', 'n_rings', 'n_ar_rings',
         'n_HBA', 'n_HBD', 'Fsp3', 'logP', 'NHOHCount', 'NOCount', 'NumHeteroatoms',
        'n_positive', '_n_negative', 'n_ring_asmbl', 'n_stereo']
a=calc_descriptors(final_tox_data)
descdf=pd.DataFrame(a, columns=descs)
descdf_approved=descdf.reset_index(drop=True)

In [ ]:
# Combine Features

features = pd.concat([final_tox_data, Morgan_fingerprint_table, MACCSfingerprint_table, descdf_approved, Mordred_table], axis=1)
test_data_features  = features[features['Data']=="DILI"].reset_index(drop=True)
train_data_features = features[features['Data']=="LivTox"].reset_index(drop=True)

test_data_features_file  = os.path.join(data_dir, 'Test_data_features.csv')
train_data_features_file = os.path.join(data_dir, 'Train_data_features.csv')

test_data_features.to_csv(test_data_features_file, index=False)
train_data_features.to_csv(train_data_features_file, index=False)

train_data = train_data_features[~train_data_features.Source_rank.isin([2, 4, 9, 10, 12])].reset_index(drop=True)
train_data = train_data[~train_data.InChIKey14.isin(test_data_features.InChIKey14.to_list())].reset_index(drop=True)

In [ ]:
# Final Checks

print(test_data_features.TOXICITY.value_counts())

for i in test_data_features.Source_rank.unique():
    print(i)
    print(test_data_features[test_data_features["Source_rank"]==i]["TOXICITY"].value_counts())
    print(len(test_data_features[test_data_features["Source_rank"]==i]))

for Source_rank in train_data.Source_rank.unique():
    print(Source_rank)
    print(len(train_data[train_data["Source_rank"] == Source_rank]))
    print(train_data[train_data["Source_rank"] == Source_rank].TOXICITY.value_counts())

for Source_rank in test_data_features.Source_rank.unique():
    print(Source_rank)
    print(len(test_data_features[test_data_features["Source_rank"] == Source_rank]))
    print(test_data_features[test_data_features["Source_rank"] == Source_rank].TOXICITY.value_counts())